# SDPO 训练快速示例

基于 `eval/core_sdpo.py` 与 `ARChitects/architect_sdpo.py`，演示如何从候选采样报告构造 SDPO 样本并跑一个训练 step。

In [1]:
import os
print(os.getcwd())
os.chdir("/data/coding/ARC")

/data/coding


In [2]:
from dataclasses import dataclass
import torch
from ARChitects.architect_sdpo import SDPOConfig, load_model_and_tokenizer, load_sdpo_dataset, build_dataloader, SDPOTrainer

# 可根据实际路径调整
config = SDPOConfig(
    model_path="outputs/arc_lora_sft_C/lora_merged_model",
    tokenizer_path="outputs/arc_lora_sft_C/lora_merged_model",
    report_path="outputs/architect_eval/report_architect_dfs_evaluation.json",  # core_cand 评估生成的报告
    dataset_root="data/evaluation",    # ARC 评估集目录
    batch_size=2, # 2
    learning_rate=5e-6,
    max_steps=100,
)
device = torch.device(config.device)
print(config)

/data/miniconda/envs/torch/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


SDPOConfig(model_path='outputs/arc_lora_sft_C/lora_merged_model', tokenizer_path='outputs/arc_lora_sft_C/lora_merged_model', report_path='outputs/architect_eval/report_architect_dfs_evaluation.json', dataset_root='data/evaluation', batch_size=2, learning_rate=5e-06, weight_decay=0.01, grad_clip=1.0, warmup_steps=10, max_steps=100, loss_type='js', top_k=100, device='cuda')


In [3]:
model, tokenizer = load_model_and_tokenizer(config)
model.config.use_cache = False
dataset = load_sdpo_dataset(config)
print(f"trainable examples: {len(dataset)}")

dataloader = build_dataloader(
    tokenizer=tokenizer,
    dataset=dataset,
    device=device,
    batch_size=config.batch_size,
)
trainer = SDPOTrainer(model, tokenizer, config)

Loading weights: 100%|█████████████████████████████████████████████████████| 398/398 [00:00<00:00, 443.49it/s, Materializing param=model.norm.weight]


[sdpo] model/tokenizer loaded from outputs/arc_lora_sft_C/lora_merged_model
[sdpo] load report: outputs/architect_eval/report_architect_dfs_evaluation.json
[sdpo] scanning record #20 task=0e671a1a
[sdpo] scanning record #40 task=195ba7dc
[sdpo] scanning record #60 task=22a4bbc2
[sdpo] scanning record #80 task=31adaf00
[sdpo] scanning record #100 task=423a55dc
[sdpo] scanning record #120 task=4ff4c9da
[sdpo] scanning record #140 task=5a5a2103
[sdpo] scanning record #160 task=67b4a34d
[sdpo] scanning record #180 task=759f3fd3
[sdpo] scanning record #200 task=84f2aca1
[sdpo] scanning record #220 task=929ab4e9
[sdpo] scanning record #240 task=9bebae7a
[sdpo] scanning record #260 task=ac0c5833
[sdpo] scanning record #280 task=b942fd60
[sdpo] scanning record #300 task=c658a4bd
[sdpo] scanning record #320 task=d282b262
[sdpo] scanning record #340 task=e133d23d
[sdpo] scanning record #360 task=e78887d1
[sdpo] scanning record #380 task=f21745ec
[sdpo] scanning record #400 task=ff72ca3e
[sdpo] b

In [4]:
# 跑一个 step 演示
batch = next(iter(dataloader))
loss = trainer.train_step(batch)
print({"sdpo_loss": loss})


[sdpo] forward batch: seq=4274 loss=0.0518
[sdpo] train_step loss=0.0518
{'sdpo_loss': 0.0517578125}


In [ ]:
# 可选：快速验证教师/学生对 wrong_answer 的 logprob 提升
# metrics = trainer.quick_eval(batch)

In [4]:
from torch.amp.autocast_mode import autocast
from torch.amp.grad_scaler import GradScaler
from itertools import cycle
from eval.core_sdpo import sdpo_forward

scaler = GradScaler()

def train_loop(
    trainer: SDPOTrainer,
    train_loader,
    *,
    valid_loader=None,
    max_steps=None,
    log_every=10,
    eval_every=100,
):
    # 确保 train_step 只做前向返回 loss，不做 backward/step
    max_steps = max_steps or trainer.config.max_steps
    step = 0
    train_iter = cycle(train_loader)

    optimizer = trainer.optimizer
    scheduler = trainer.lr_scheduler

    while step < max_steps:
        batch = next(train_iter)
        optimizer.zero_grad(set_to_none=True)
        with autocast(device_type=trainer.config.device):
            loss, _ = sdpo_forward(          # 直接调用 sdpo_forward，或改成 trainer.forward(batch)
                trainer.model,
                trainer.tokenizer,
                batch,
                loss_type=trainer.config.loss_type,
                top_k=trainer.config.top_k,
            )
        scaler.scale(loss).backward()
        if trainer.config.grad_clip is not None:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(trainer.model.parameters(), trainer.config.grad_clip)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        step += 1

        if step % log_every == 0:
            print(f"[sdpo] step {step}/{max_steps} train_loss={loss.item():.4f}")

        if valid_loader is not None and step % eval_every == 0:
            try:
                vbatch = next(iter(valid_loader))
            except StopIteration:
                vbatch = next(iter(valid_loader))
            with torch.no_grad(), autocast(device_type=trainer.config.device):
                v_loss, _ = sdpo_forward(
                    trainer.model,
                    trainer.tokenizer,
                    vbatch,
                    loss_type=trainer.config.loss_type,
                    top_k=trainer.config.top_k,
                )
            print(f"[sdpo] eval step {step}: loss={v_loss.item():.4f}")

    print(f"[sdpo] training done: steps={step}")
    return trainer

In [ ]:
valid_loader =

In [5]:
# 调整 max_steps/log_every/eval_every 后直接运行

_ = train_loop(
    trainer,
    dataloader,
    valid_loader=None,  # 若有验证集 dataloader 可放这里
    max_steps=20,
    log_every=5,
    eval_every=25,
)


OutOfMemoryError: CUDA out of memory. Tried to allocate 330.00 MiB. GPU 0 has a total capacity of 39.50 GiB of which 295.38 MiB is free. Including non-PyTorch memory, this process has 0 bytes memory in use. Of the allocated memory 35.75 GiB is allocated by PyTorch, and 2.95 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)